# Sinyal Dini Zoonosis dengan Model SEIR

**ID proyek:** `O005-LEGA-V101-PRJ02`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Informasi apa tentang laju pertumbuhan awal yang dapat dipulihkan dari pengamatan wabah zoonotik yang jarang dan berisik?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082202
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Populasi homogen dan tertutup mengikuti kompartemen SEIR; parameter tetap selama jendela awal; pengamatan sintetis merupakan prevalensi infeksi dengan gangguan kecil.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
beta, sigma, gamma = 0.72, 1.0 / 4.5, 1.0 / 6.0
R0 = beta / gamma
t_eval = np.linspace(0.0, 120.0, 481)

def seir_rhs(t, y):
    S, E, I, R = y
    return [-beta * S * I, beta * S * I - sigma * E, sigma * E - gamma * I, gamma * I]

solution = solve_ivp(seir_rhs, (t_eval[0], t_eval[-1]), [0.997, 0.002, 0.001, 0.0], t_eval=t_eval, rtol=1e-9, atol=1e-11)
S, E, I, R = solution.y
observation_days = np.arange(0, 121, 5)
I_observed = np.clip(np.interp(observation_days, t_eval, I) + rng.normal(0.0, 0.00035, observation_days.size), 0.0, None)


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert solution.success and R0 > 1.0
np.testing.assert_allclose(S + E + I + R, 1.0, atol=2e-8)
assert np.min(solution.y) > -1e-10
assert float(t_eval[np.argmax(I)]) > 5.0 and float(np.max(I)) > I[0]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for values, label in [(S, "S"), (E, "E"), (I, "I"), (R, "R")]:
    ax.plot(t_eval, values, label=label)
ax.scatter(observation_days, I_observed, s=18, color="black", label="pengamatan I sintetis")
ax.set(xlabel="hari", ylabel="fraksi populasi", title="Lintasan SEIR dan pengamatan dini")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Tidak ada struktur umur, pelaporan tertunda, limpahan berulang dari hewan, perubahan perilaku, atau variasi spasial.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
